# Improve detection on local camera images

Run the cells in order in a fresh Colab session with a GPU. The training uses the existing reviewed train split, including team-collected training images and empty frames. Selection uses the existing CCT20 selection-validation split. Team test images are evaluated only after training and validation. Outputs go into a new versioned folder in Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, tempfile

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path(tempfile.mkdtemp()) / 'project'
subprocess.run(['git','clone','--quiet','--depth','1','--branch','project',
                'https://github.com/salehdev3/Capstone_5.git',str(PROJECT)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')], check=True)
sys.path.insert(0,str(PROJECT/'runtime'))

import torch
import pandas as pd
from ultralytics import YOLO
from common import read, save, sha
from live import RUN, initialize, execute, team_test_cases
from deployment import GuidedPipeline
from IPython.display import display, Image

if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then rerun.')
policy_path = ROOT / RUN / 'policy.json'
decision_path = ROOT / RUN / 'decision.json'
policy = read(policy_path)
assert sha(policy_path) == read(decision_path)['policy_sha256'], 'Integrated policy changed.'
source_detector = Path(policy['detector']['path'])
assert source_detector.is_file() and sha(source_detector) == policy['detector']['sha256']
OUT = ROOT / 'phase2_results/local_detection_improvement' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT.mkdir(parents=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Results folder:',OUT)


In [ ]:
# Use the existing image and YOLO annotation release.
manifest = read(ROOT / 'detection_manifest.json')
train = [r for r in manifest if r['split'] == 'train']
val = [r for r in manifest if r['split'] == 'val_select']
local_train = [r for r in train if r.get('source') == 'team_collected']
if not local_train:
    raise ValueError('No team-collected training frames found in detection_manifest.json.')
if not train or not val:
    raise ValueError('Training and selection-validation splits are required.')
for row in train + val:
    image = ROOT / 'images' / (row['id'] + '.jpg')
    label = ROOT / 'labels' / (row['id'] + '.txt')
    if not image.is_file() or not label.is_file():
        raise FileNotFoundError(f'Missing reviewed image or YOLO label: {image} / {label}')

# Keep the complete train distribution. Repeat the small local training subset.
train_paths = [str(ROOT / 'images' / (r['id'] + '.jpg')) for r in train]
train_paths += [str(ROOT / 'images' / (r['id'] + '.jpg')) for r in local_train for _ in range(3)]
val_paths = [str(ROOT / 'images' / (r['id'] + '.jpg')) for r in val]
(OUT / 'train.txt').write_text('\n'.join(train_paths) + '\n')
(OUT / 'val.txt').write_text('\n'.join(val_paths) + '\n')
data = dict(path=str(OUT), train=str(OUT/'train.txt'), val=str(OUT/'val.txt'), names={0:'animal'})
save(OUT/'data.yaml', data)
save(OUT/'protocol.json', dict(train_unique=len(train),local_train_unique=len(local_train),
    local_repeats=4, val_images=len(val), team_test_used_for_training=False,
    train_ids=[r['id'] for r in train], val_ids=[r['id'] for r in val],
    parent_policy_sha256=sha(policy_path), parent_detector_sha256=sha(source_detector)))
print('Train frames:',len(train),'| local train:',len(local_train),'| selection-validation:',len(val))
print('Local training empty frames:',sum(not r['animal'] for r in local_train))


In [ ]:
# Fine-tune the existing detector. The run saves best.pt and last.pt in Drive.
model = YOLO(str(source_detector))
model.train(data=str(OUT/'data.yaml'), imgsz=policy['detector']['size'],
            epochs=30, patience=8, batch=8, device=0, workers=2,
            seed=42, project=str(OUT), name='detector', exist_ok=False,
            close_mosaic=5, save=True, plots=True)
candidate = OUT/'detector/weights/best.pt'
assert candidate.is_file()
print('Candidate saved:',candidate)


In [ ]:
# Compare detector AP on the same selection-validation images.
def validate_detector(path, name):
    model = YOLO(str(path))
    result = model.val(data=str(OUT/'data.yaml'), split='val', imgsz=policy['detector']['size'],
                       device=0, batch=8, workers=2, conf=0.001, plots=False,
                       project=str(OUT), name=name, exist_ok=False)
    return dict(model=name, map50=float(result.box.map50), map50_95=float(result.box.map),
                precision=float(result.box.mp), recall=float(result.box.mr))

comparison = [validate_detector(source_detector,'baseline_validation'),
              validate_detector(candidate,'candidate_validation')]
save(OUT/'validation_comparison.json',comparison)
display(pd.DataFrame(comparison))
print('Compare both rows before selecting a model. Team-test results are historical development evidence.')


In [ ]:
# Compare both complete pipelines on the same team test frames.
# Existing weights and the deployed policy remain unchanged.
import copy
batch = team_test_cases(ROOT)
baseline, _ = initialize(ROOT)
candidate_policy = copy.deepcopy(policy)
candidate_policy['detector']['path'] = str(candidate)
candidate_policy['detector']['sha256'] = sha(candidate)
candidate_policy['parent_policy_sha256'] = sha(policy_path)
save(OUT/'candidate_policy.json',candidate_policy)
new_pipeline = GuidedPipeline(ROOT,Path(tempfile.mkdtemp(prefix='local_candidate_')),OUT/'candidate_policy.json')

for name, pipe in [('baseline',baseline),('candidate',new_pipeline)]:
    print('Running',name,'on',len(batch),'historical team-test images...')
    folder, result = execute(pipe,batch,OUT/name)
    print(name,'saved:',folder)
    display(pd.DataFrame([{'id':f['id'], 'count':f['count'],
                           'seconds':f['timing']['total'],
                           'species':', '.join(o['species'] for o in f['objects'])}
                          for f in result['frames']]))
    for image in sorted(folder.glob('frame_*.jpg')):
        display(Image(filename=str(image),width=480))
print('Upload validation_comparison.json, protocol.json, and both service_test.json files.')


## Resume candidate evaluation

Run the next cell on its own in a fresh Colab session to resume the latest completed candidate. This loads the saved weights and does not train again.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, tempfile
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path(tempfile.mkdtemp()) / 'project'
subprocess.run(['git','clone','--quiet','--depth','1','--branch','project',
                'https://github.com/salehdev3/Capstone_5.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
sys.path.insert(0,str(PROJECT/'runtime'))
from common import read, sha
from deployment import GuidedPipeline
from live import execute, team_test_cases
from IPython.display import display, Image
import pandas as pd

base = ROOT/'phase2_results/local_detection_improvement'
runs = sorted((p for p in base.glob('*') if (p/'candidate_policy.json').is_file()
               and (p/'validation_comparison.json').is_file()), key=lambda p:p.name)
if not runs:
    raise FileNotFoundError('No completed candidate with a policy and validation comparison found.')
OUT = runs[-1]
policy_path = OUT/'candidate_policy.json'
policy = read(policy_path)
weights = Path(policy['detector']['path'])
if not weights.is_file() or sha(weights) != policy['detector']['sha256']:
    raise ValueError('Candidate detector checkpoint is missing or its hash changed.')
print('Resuming candidate:',OUT)
print('Validation comparison:')
display(pd.DataFrame(read(OUT/'validation_comparison.json')))
pipeline = GuidedPipeline(ROOT,Path(tempfile.mkdtemp(prefix='local_candidate_')),policy_path)
batch = team_test_cases(ROOT)
folder, result = execute(pipeline,batch,OUT/'candidate')
display(pd.DataFrame([{'id':f['id'], 'count':f['count'], 'seconds':f['timing']['total'],
                      'species':', '.join(o['species'] for o in f['objects'])}
                     for f in result['frames']]))
for path in sorted(folder.glob('frame_*.jpg')):
    display(Image(filename=str(path),width=480))
print('Candidate output:',folder)
print('Upload service_test.json and Demo_Fallback.mp4 from this folder.')
